# Credit Card Case Study - Complete Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
customer = pd.read_csv(r"Customer Acqusition.csv")
repayment = pd.read_csv(r"Repayment.csv")
spend = pd.read_csv(r"spend.csv")

### 1a. Clean Age - Replace age < 18 with mean

In [ ]:
customer['Age'] = customer['Age'].astype('int')
mean_age = customer.loc[customer['Age'] >= 18, 'Age'].mean()
customer['Age'] = np.where(customer['Age'] < 18, mean_age, customer['Age'])

### 1b. Clean Spend - Replace amount > limit with 50% of limit

In [ ]:
cust_spend = pd.merge(left=spend, right=customer, on='Customer')
spend.loc[cust_spend['Amount'] > cust_spend['Limit'], 'Amount'] = cust_spend.loc[cust_spend['Amount'] > cust_spend['Limit'], 'Limit'] * 0.5
cust_spend = pd.merge(left=spend, right=customer, on='Customer')

### 1c. Clean Repayment - Replace repayment > limit with limit

In [ ]:
repayment = repayment.iloc[:, :4].dropna(how='all').reset_index(drop=True)
repayment['Month'] = pd.to_datetime(repayment['Month'], format='%d-%b-%y')
cust_repay = pd.merge(left=repayment, right=customer, on='Customer')
mask = cust_repay['Amount'] > cust_repay['Limit']
repayment.loc[mask, 'Amount'] = cust_repay.loc[mask, 'Limit']

### Convert Month columns to datetime

In [ ]:
spend['Month'] = pd.to_datetime(spend['Month'], format='%d-%b-%y')
cust_spend['Month'] = pd.to_datetime(cust_spend['Month'], format='%d-%b-%y')

### 2a. How many distinct customers exist?

In [ ]:
print(customer['Customer'].nunique())

### 2b. How many distinct categories exist?

In [ ]:
print(spend['Type'].nunique())
print(spend['Type'].unique())

### 2c. Average monthly spend by customers

In [ ]:
pd.DataFrame(spend.groupby('Customer')['Amount'].mean()).sort_values(by='Amount', ascending=False)

### 2d. Average monthly repayment by customers

In [ ]:
pd.DataFrame(repayment.groupby('Customer')['Amount'].mean()).sort_values(by='Amount', ascending=False)

### 2e. Monthly bank profit at 2.9% interest rate

In [ ]:
monthly_repayment = repayment.groupby(repayment['Month'].dt.month)['Amount'].sum()
monthly_spend = spend.groupby(spend['Month'].dt.month)['Amount'].sum()
monthly_profit = monthly_repayment - monthly_spend
bank_profit = monthly_profit.apply(lambda x: x * 0.029 if x > 0 else 0)
print(bank_profit)

### 2f. Top 5 product types

In [ ]:
cust_spend.groupby('Type')['Amount'].sum().sort_values(ascending=False).head(5)

### 2g. City with maximum spend

In [ ]:
cust_spend.groupby('City')['Amount'].sum().sort_values(ascending=False).head(1)

### 2h. Age group with highest spending

In [ ]:
cust_spend.groupby('Age')['Amount'].sum().sort_values(ascending=False).head(1)

### 2i. Top 10 customers by repayment

In [ ]:
pd.DataFrame(repayment.groupby('Customer')['Amount'].sum()).sort_values(by='Amount', ascending=False).head(10)

### 3. City wise spend on each product on yearly basis

In [ ]:
result = cust_spend.groupby([cust_spend['Month'].dt.year, 'Product', 'City'])['Amount'].sum().reset_index()
result.rename(columns={'Month': 'Year'}, inplace=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, year in enumerate(result['Year'].unique()):
    data = result[result['Year'] == year]
    pivot = pd.crosstab(index=data['City'], columns=data['Product'], values=data['Amount'], aggfunc='sum')
    pivot.plot(kind='bar', ax=axes[i], title=f'Year: {year}')
    axes[i].set_xlabel('City')
    axes[i].set_ylabel('Amount')
    axes[i].tick_params(axis='x', rotation=45)
    for container in axes[i].containers:
        axes[i].bar_label(container, fmt='%.0f', fontsize=6)

plt.tight_layout()
plt.show()

### 4a. Monthly comparison of total spends city wise

In [ ]:
result1 = pd.crosstab(index=cust_spend['Month'].dt.month, columns=cust_spend['City'], values=cust_spend['Amount'], aggfunc='sum')
result1.plot(kind='line', marker='o', markersize=3, figsize=(12, 5), title='Monthly Comparison of Spend City Wise')
plt.xlabel('Months')
plt.ylabel('Amount')
plt.xticks(ticks=range(1, 13), labels=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.tight_layout()
plt.show()

### 4b. Comparison of yearly spend on air tickets

In [ ]:
air_type = cust_spend.loc[cust_spend['Type'] == 'AIR TICKET']
result2 = air_type.groupby(air_type['Month'].dt.year)['Amount'].sum().reset_index()
result2.rename(columns={'Month': 'Year'}, inplace=True)

ax = result2.plot(x='Year', y='Amount', kind='bar', title='Yearly Spend on Air Tickets', figsize=(8, 5))
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', fontsize=9)
ax.set_xlabel('Years')
ax.set_ylabel('Amount')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 4c. Monthly spend for each product (seasonality)

In [ ]:
result3 = pd.crosstab(index=cust_spend['Month'].dt.month, columns=cust_spend['Product'], values=cust_spend['Amount'], aggfunc='sum')
result3.plot(kind='line', marker='o', markersize=4, figsize=(12, 5), title='Monthly Spend for each Product')
plt.xlabel('Months')
plt.ylabel('Amount')
plt.xticks(ticks=range(1, 13), labels=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.tight_layout()
plt.show()

### 5. Top 10 customers per city by repayment (User Defined Function)

In [ ]:
cust_repay = pd.merge(left=repayment, right=customer, on='Customer')

def func1(product, time_period):
    df = cust_repay.loc[cust_repay['Product'] == product]
    
    if time_period > 12:  # year
        df = df.loc[df['Month'].dt.year == time_period]
    else:  # month
        df = df.loc[df['Month'].dt.month == time_period]
    
    df1 = df.groupby(['City', 'Customer'])['Amount'].sum().reset_index()
    df2 = df1.sort_values(['City', 'Amount'], ascending=[True, False])
    df2 = df2.groupby('City').head(10).reset_index(drop=True)
    
    return df2

# Example usage
func1('Gold', 2004)    # top 10 per city for Gold in year 2004
# func1('Silver', 6)  # top 10 per city for Silver in month June